# Validator Pipeline Manual Test Suite

This notebook collects the manual test snippets previously outlined for the core validator pipeline components (competition management, scoring, EMA aggregation, persistence, API flows, scheduler logic, and weight setter). Run the cells sequentially inside Google Colab or a local environment after installing the project's dependencies.

## 1. Environment Bootstrap

Clone the repository, move into it, and install dependencies along with runtime extras that are required for validator simulations.

In [ ]:
!git clone https://github.com/tensorlink-dev/epochor.git
%cd epochor
!pip install -r requirements.txt
!pip install bittensor fastapi uvicorn apscheduler safetensors

> **Note:** Skip the cloning step if you already have the repository mounted.

## 2. Competition Schedule & Dataset Sanity

Validate that the `CompetitionManager` produces competitions with datasets that match the configured schedule, and inspect the tensor shapes provided by the synthetic benchmarker.

In [ ]:
import threading, tempfile, torch
from competitions.competitions import COMPETITION_SCHEDULE_BY_BLOCK
from neurons.validator.competition_manager import CompetitionManager
from neurons.validator.state import ValidatorState

class DummyMetagraph:
    def __init__(self, n):
        self.uids = list(range(n))
        self.hotkeys = [f"dummy_hotkey_{i}" for i in range(n)]
        self.n = n

metagraph = DummyMetagraph(4)
state = ValidatorState(metagraph, tempfile.mkdtemp(), threading.RLock())
manager = CompetitionManager(state)

competition = manager.get_next_competition(global_step=0, block=0)
tasks, samples = manager.load_data_for_competition(competition, seed=123)

print(competition.id, competition.reward_percentage)
print(len(tasks), len(samples))
print({k: v.shape for k, v in samples[0][0].items() if torch.is_tensor(v)})

## 3. Miner Submission Contract Smoke Test

Create a toy submission that implements the `MinerSubmissionProtocol` and ensure the validator-supplied batch produces a valid training step.

In [ ]:
from epochor.training.validator_contract import MinerSubmissionProtocol
from torch import nn
import torch.optim as optim

class ToySubmission(MinerSubmissionProtocol):
    def build_model(self, cfg):
        return nn.GRU(input_size=1, hidden_size=8, batch_first=True)
    def build_optimizer(self, model, cfg):
        return optim.Adam(model.parameters(), lr=cfg.get("lr", 1e-3))
    def train_step(self, model, batch, optimizer, step_idx, cfg):
        model.train()
        optimizer.zero_grad()
        preds, _ = model(batch["inputs_padded"].unsqueeze(-1))
        loss = (preds.squeeze(-1) - batch["targets_padded"]).pow(2).mean()
        loss.backward()
        optimizer.step()
        return {"loss": loss.item(), "step": step_idx}

submission = ToySubmission()
cfg = {"lr": 1e-3}
model = submission.build_model(cfg)
opt = submission.build_optimizer(model, cfg)
metrics = submission.train_step(model, samples[0][0], opt, 0, cfg)
print(metrics)

## 4. Evaluation Scoring Pipeline Check

Use the reference autoregressive model with the scoring helper to ensure end-to-end evaluation returns finite metrics and detailed diagnostics.

In [ ]:
from epochor.model.model_constraints import SimpleAutoregressiveModel, SimpleTimeSeriesConfig
from epochor.validation.validation import score_time_series_model

ts_model = SimpleAutoregressiveModel(SimpleTimeSeriesConfig())
score, details = score_time_series_model(ts_model, samples, tasks, device="cpu", task_or_seed=123)
print("Mean loss:", score)
print("Available detail keys:", list(details.keys()))

## 5. Scoring & Retention Simulation

Exercise the scoring service to ensure sample minima, EMA updates, and keeper selection behave as expected.

In [ ]:
import numpy as np, types
from neurons.validator.scoring_service import ScoringService, PerUIDEvalState

dummy_config = types.SimpleNamespace(sample_min=2)
state.ema_tracker.clear_all()
uids = [0, 1, 2, 3]
uid_to_state = {
    uid: PerUIDEvalState(
        hotkey=metagraph.hotkeys[uid],
        score_details={"flat_evaluation": types.SimpleNamespace(raw_score=np.random.rand(5))}
    )
    for uid in uids
}
scorer = ScoringService(state, metagraph, dummy_config)
scores, keepers = scorer.process_scores_and_update_weights(uids, uid_to_state, competition, cur_block=123)
print(scores["final_scores_dict"])
print("Retained UIDs:", keepers)
print("Stored EMA weights shape:", state.ema_tracker.get_competition_weights(competition.id).shape)

## 6. Subnet Weight Aggregation

Combine EMA snapshots across competitions to verify weights are normalized and align with the tracked UIDs.

In [ ]:
weights = state.ema_tracker.subnet_weights([competition], min_comp_weight_threshold=1e-3)
print(weights.sum().item(), weights[:len(uids)])

## 7. Validator State Persistence

Persist the validator state to disk and reload it to ensure queue tracking and EMA history survive restarts.

In [ ]:
state.save()
state.load()
print(state.get_pending_and_current_uid_counts())

## 8. FastAPI Platform Flow

Spin up the FastAPI app with an in-memory SQLite database and walk through miner submission, lease allocation, result submission, heartbeat, and weight retrieval.

In [ ]:
import os
from fastapi.testclient import TestClient
from api.config import Settings
from api import database
from api.main import create_app

os.environ["EPOCHOR_DATABASE_URL"] = "sqlite+pysqlite:///:memory:"
database.configure_engine(os.environ["EPOCHOR_DATABASE_URL"])
database.init_db()
app = create_app()
client = TestClient(app)

settings = Settings()
settings.allowed_miner_hotkeys.append("miner-hotkey")
settings.allowed_validator_hotkeys.append("validator-hotkey")

resp = client.post("/miner/submit", json={"hotkey": "miner-hotkey", "model_code_url": "https://hf/model"})
submission = resp.json()
print("Submission:", submission)

lease = client.post("/validator/request-training-job", json={"validator_hotkey": "validator-hotkey"}).json()
print("Lease:", lease)

client.post("/validator/submit-results", json={
    "submission_id": lease["submission_id"],
    "model_id": lease["model_id"],
    "new_score": 0.42,
    "validator_hotkey": "validator-hotkey",
    "meta": {"notes": "manual check"}
})

client.post("/validator/heartbeat", json={"validator_hotkey": "validator-hotkey"})
weights = client.get("/scoring/weights").json()
print("Weights:", weights)

## 9. Scheduler Promotion Logic

Seed sample submissions, run promotion/cleanup helpers, and inspect the resulting pool assignments.

In [ ]:
from api.competition_scheduler import promote_to_medium, promote_to_final, select_winner, cleanup_rejected
from api.models import ModelSubmission, SubmissionPool
from api.database import session_scope

with session_scope() as session:
    for i, score in enumerate([0.3, 0.5, None, 0.1]):
        session.add(ModelSubmission(
            hotkey=f"miner-{i}",
            model_id=f"model-{i}",
            current_pool=SubmissionPool.SHALLOW,
            current_score=score,
        ))

with session_scope() as session:
    promote_to_medium(session, settings)
    promote_to_final(session, settings)
    select_winner(session)
    cleanup_rejected(session)

with session_scope() as session:
    pools = {sub.model_id: sub.current_pool for sub in session.query(ModelSubmission).all()}
print(pools)

## 10. Weight Setter Dry Run

Ensure the weight setter converts tensors and interacts with the (stubbed) subtensor client without deadlocking on the metagraph lock.

In [ ]:
import torch, types, asyncio
from neurons.validator.weight_setter import WeightSetter

class DummySubtensor:
    def set_weights(self, **kwargs):
        print("Weights pushed", kwargs["weights"][:4])
        return True, "ok"

weights_tensor = torch.zeros(len(metagraph.uids))
setter = WeightSetter(
    subtensor=DummySubtensor(),
    wallet=None,
    netuid=1,
    metagraph=metagraph,
    weights=weights_tensor,
    metagraph_lock=threading.RLock(),
    api_base_url="http://localhost:8000"
)
setter._pull_latest_weights()
asyncio.run(setter._set_weights())

## 11. Additional Manual Checks

- Interact with `ModelTracker` APIs to confirm training history management.
- Introduce malformed dataset parameters to observe error handling in the competition manager.
- Execute the sandbox template inside Docker for parity with production (optional).